## Import Library

In [1]:
import requests
import csv
from bs4 import BeautifulSoup

In [2]:
# Satu halaman MAL hanya berisi 50 anime.
# Untuk >= 200 data, ambil 4 halaman: limit=0, 50, 100, 150.
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

base_url = "https://myanimelist.net/topanime.php?limit={}"
pages_html = []

for limit in [0, 50, 100, 150]:
    url = base_url.format(limit)
    response = requests.get(url, headers=headers)

    print(f"URL: {url} -> status: {response.status_code}")

    if response.status_code != 200:
        raise Exception(f"Gagal mengambil {url}. Status: {response.status_code}")

    pages_html.append(response.text)

print(f"\nTotal halaman berhasil diambil: {len(pages_html)}")

URL: https://myanimelist.net/topanime.php?limit=0 -> status: 200
URL: https://myanimelist.net/topanime.php?limit=50 -> status: 200
URL: https://myanimelist.net/topanime.php?limit=100 -> status: 200
URL: https://myanimelist.net/topanime.php?limit=150 -> status: 200

Total halaman berhasil diambil: 4


In [3]:
soups = [BeautifulSoup(html, "html.parser") for html in pages_html]

anime_blocks = []
for soup in soups:
    rows = soup.find_all("tr", class_="ranking-list")
    anime_blocks.extend(rows)

print(f"Jumlah baris anime yang ditemukan: {len(anime_blocks)}")

if len(anime_blocks) < 200:
    print("Data kurang dari 200. Tambahkan limit halaman lain.")
else:
    print("Jumlah data memenuhi syarat minimal 200.")

Jumlah baris anime yang ditemukan: 200
Jumlah data memenuhi syarat minimal 200.


In [4]:
import re

result = []

for block in anime_blocks:
    # --- Judul (kategori) ---
    title_element = block.find("h3", class_="anime_ranking_h3")
    title_name = title_element.get_text(strip=True) if title_element else None

    # --- Information (tipe + episode + members) ---
    info_element = block.find("div", class_="information")
    info_text = info_element.get_text(" ", strip=True) if info_element else ""

    anime_type = None
    episodes = None
    members_value = None

    if info_text:
        # Tipe & episode: "TV (28 eps)"
        if "(" in info_text and ")" in info_text:
            anime_type = info_text.split("(")[0].strip()
            inside = info_text.split("(")[1].split(")")[0]
            if "eps" in inside.lower():
                episodes = inside.lower().replace("eps", "").strip()

        # Members: ambil angka sebelum kata "members", buang koma
        match = re.search(r"([\d,]+)\s*members", info_text)
        if match:
            members_value = match.group(1).replace(",", "")

    # --- Skor (angka) ---
    score_element = block.find("span", class_="score-label")
    score_value = score_element.get_text(strip=True) if score_element else None

    result.append({
        "title": title_name,
        "type": anime_type,
        "episodes": episodes,
        "score": score_value,
        "members": members_value,
    })

# sanity check
for r in result[:5]:
    print(r)

{'title': 'Sousou no Frieren', 'type': 'TV', 'episodes': '28', 'score': '9.24', 'members': '1541261'}
{'title': 'Fullmetal Alchemist: Brotherhood', 'type': 'TV', 'episodes': '64', 'score': '9.11', 'members': '3753854'}
{'title': 'Steins;Gate', 'type': 'TV', 'episodes': '24', 'score': '9.07', 'members': '2865402'}
{'title': 'Re:Zero kara Hajimeru Isekai Seikatsu 4th Season', 'type': 'TV', 'episodes': '19', 'score': '9.06', 'members': '368392'}
{'title': 'Shingeki no Kyojin Season 3 Part 2', 'type': 'TV', 'episodes': '10', 'score': '9.06', 'members': '2655943'}


In [5]:
for item in result:
    print(
        f"Title: {item['title']} - "
        f"Type: {item['type']} - "
        f"Episodes: {item['episodes']} - "
        f"Score: {item['score']} - "
        f"Members: {item['members']}"
    )

Title: Sousou no Frieren - Type: TV - Episodes: 28 - Score: 9.24 - Members: 1541261
Title: Fullmetal Alchemist: Brotherhood - Type: TV - Episodes: 64 - Score: 9.11 - Members: 3753854
Title: Steins;Gate - Type: TV - Episodes: 24 - Score: 9.07 - Members: 2865402
Title: Re:Zero kara Hajimeru Isekai Seikatsu 4th Season - Type: TV - Episodes: 19 - Score: 9.06 - Members: 368392
Title: Shingeki no Kyojin Season 3 Part 2 - Type: TV - Episodes: 10 - Score: 9.06 - Members: 2655943
Title: Chainsaw Man Movie: Reze-hen - Type: Movie - Episodes: 1 - Score: 9.05 - Members: 573671
Title: Gintama: The Final - Type: Movie - Episodes: 1 - Score: 9.05 - Members: 190079
Title: Gintama° - Type: TV - Episodes: 51 - Score: 9.05 - Members: 702029
Title: Hunter x Hunter (2011) - Type: TV - Episodes: 148 - Score: 9.03 - Members: 3260221
Title: Steel Ball Run: JoJo no Kimyou na Bouken - Type: ONA - Episodes: ? - Score: 9.03 - Members: 255228
Title: Ginga Eiyuu Densetsu - Type: OVA - Episodes: 110 - Score: 9.02 - 

In [6]:
with open("anime_top.csv", "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["title", "type", "episodes", "score", "members"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for item in result:
        writer.writerow(item)

print("Data disimpan ke anime_top.csv")

Data disimpan ke anime_top.csv


In [7]:
import pandas as pd

db = pd.read_csv("anime_top.csv")

# Kolom angka hasil scraping masih berupa teks.
# episodes: bisa berisi "?" untuk anime ongoing -> jadi NaN
# score: desimal biasa
# members: sudah dibersihkan dari koma di Cell 6
# Maftuh Faisal_202431163
db["episodes"] = pd.to_numeric(db["episodes"], errors="coerce")
db["score"] = pd.to_numeric(db["score"], errors="coerce")
db["members"] = pd.to_numeric(db["members"], errors="coerce")

print(db.dtypes)

# Simpan hasil bersih
db.to_csv("anime_top_clean.csv", index=False)
print("Data bersih disimpan ke anime_top_clean.csv")

title           str
type            str
episodes    float64
score       float64
members       int64
dtype: object
Data bersih disimpan ke anime_top_clean.csv


In [8]:
db

,title,type,episodes,score,members
0,Sousou no Frieren,TV,28.0,9.24,1541261
1,Fullmetal Alchemist: Brotherhood,TV,64.0,9.11,3753854
2,Steins;Gate,TV,24.0,9.07,2865402
3,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,TV,19.0,9.06,368392
4,Shingeki no Kyojin Season 3 Part 2,TV,10.0,9.06,2655943
...,...,...,...,...,...
195,Mo Dao Zu Shi: Xian Yun Pian,ONA,8.0,8.45,121008
196,Owarimonogatari,TV,12.0,8.45,517767
197,Steins;Gate Movie: Fuka Ryouiki no Déjà vu,Movie,1.0,8.45,654379
198,Tian Guan Cifu Short Films,ONA,NaN,8.45,12858
